# Accelerated Gradient Descent: A Comparative Study

**Topics covered:** Vanilla Gradient Descent · Polyak Heavy Ball · Nesterov Accelerated Gradient

---

## Introduction

We minimise a **strongly convex quadratic**:

$$f(x) = \frac{1}{2} x^\top A x - b^\top x, \qquad A \in \mathbb{R}^{d\times d} \text{ symmetric positive definite.}$$

The unique minimiser is $x^\star = A^{-1}b$.

### Relevant quantities
| Symbol | Meaning |
|--------|---------|
| $\mu$ | smallest eigenvalue of $A$ (strong-convexity constant) |
| $L$ | largest eigenvalue of $A$ (smoothness constant) |
| $\kappa = L/\mu$ | **condition number** |

### Methods at a glance

| Method | Update rule |
|--------|-------------|
| **GD** | $x_{k+1} = x_k - \alpha \nabla f(x_k)$ |
| **Polyak Heavy Ball (HB)** | $x_{k+1} = x_k - \alpha \nabla f(x_k) + \beta(x_k - x_{k-1})$ |
| **Nesterov (NAG)** | $y_{k+1} = x_k - \alpha \nabla f(x_k)$, then $x_{k+1} = y_{k+1} + \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}(y_{k+1}-y_k)$ |

---

### Learning objectives
1. Implement the three optimisers from scratch.
2. Observe how the **condition number** $\kappa$ affects convergence speed.
3. Understand the role of the **momentum parameter** $\beta$.
4. Verify the theoretical convergence rates empirically.

## 0 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

rng = np.random.default_rng(42)

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

---
## 1 — Problem setup

We build a random PD matrix $A$ with a prescribed condition number $\kappa$, and a random right-hand side $b$.

In [ ]:
def make_problem(d: int, kappa: float, seed=42):
    """
    Build a random strongly-convex quadratic  f(x) = 0.5 x'Ax - b'x
    with condition number kappa.

    Returns
    -------
    A      : (d, d) SPD matrix
    b      : (d,)  vector
    x_star : (d,)  exact minimiser A^{-1} b
    mu     : smallest eigenvalue  (strong-convexity)
    L      : largest  eigenvalue  (smoothness)
    """
    rng = np.random.default_rng(seed)
    # Random orthonormal basis
    Q, _ = np.linalg.qr(rng.standard_normal((d, d)))
    # Eigenvalues log-spaced from 1 to kappa  (so kappa = L/mu = kappa/1)
    eigvals = np.exp(np.linspace(0, np.log(kappa), d))
    A = Q @ np.diag(eigvals) @ Q.T
    b = rng.standard_normal(d)
    x_star = np.linalg.solve(A, b)
    mu, L = eigvals.min(), eigvals.max()
    return A, b, x_star, mu, L


def f(x, A, b):
    """Objective value."""
    return 0.5 * x @ A @ x - b @ x


def grad_f(x, A, b):
    """Gradient: A x - b."""
    return A @ x - b


# Quick sanity-check
A, b, x_star, mu, L = make_problem(d=50, kappa=10)
print(f"mu={mu:.4f}  L={L:.4f}  kappa={L/mu:.2f}")
print(f"||grad f(x*)|| = {np.linalg.norm(grad_f(x_star, A, b)):.2e}  (should be ~0)")

---
## 2 — Implement the three optimisers

> **✏️  Exercise 2.1 — Vanilla Gradient Descent**
>
> Fill in the update rule inside `gradient_descent`.
> The optimal fixed step size for an $L$-smooth function is $\alpha = 1/L$.
> Return the list of objective values at every iterate.

In [ ]:
def gradient_descent(A, b, x_star, alpha, n_iter=500):
    """
    Vanilla Gradient Descent.

    Parameters
    ----------
    alpha  : step size
    n_iter : number of iterations

    Returns
    -------
    losses : list of  f(x_k) - f(x*)  for k = 0 .. n_iter
    """
    x = np.zeros_like(b)
    f_star = f(x_star, A, b)
    losses = [f(x, A, b) - f_star]

    for _ in range(n_iter):
        # ── TODO ─────────────────────────────────────────────────────────────
        # x = ???
        # ─────────────────────────────────────────────────────────────────────
        losses.append(f(x, A, b) - f_star)

    return losses

> **✏️ Exercise 2.2 — Polyak Heavy Ball**
>
> The update reads:
> $$x_{k+1} = x_k - \alpha \nabla f(x_k) + \beta (x_k - x_{k-1})$$
>
> For a quadratic the optimal parameters are:
> $$\alpha^\star = \left(\frac{2}{\sqrt{L}+\sqrt{\mu}}\right)^2, \qquad \beta^\star = \left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2$$
>
> Fill in the two-line update.

In [ ]:
def heavy_ball(A, b, x_star, alpha, beta, n_iter=500):
    """
    Polyak Heavy Ball (momentum gradient descent).

    Parameters
    ----------
    alpha : step size
    beta  : momentum coefficient  in [0, 1)
    """
    x = np.zeros_like(b)
    x_prev = x.copy()
    f_star = f(x_star, A, b)
    losses = [f(x, A, b) - f_star]

    for _ in range(n_iter):
        g = grad_f(x, A, b)
        # ── TODO ─────────────────────────────────────────────────────────────
        # x_new = ???   (use x, x_prev, g, alpha, beta)
        # x_prev = ???
        # x     = ???
        # ─────────────────────────────────────────────────────────────────────
        losses.append(f(x, A, b) - f_star)

    return losses

> **✏️ Exercise 2.3 — Nesterov Accelerated Gradient (NAG)**
>
> NAG introduces a *look-ahead* point $y_k$ and performs the gradient step there:
>
> $$\text{(gradient step)} \quad y_{k+1} = x_k - \alpha \nabla f(x_k)$$
> $$\text{(momentum step)} \quad x_{k+1} = y_{k+1} + \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\bigl(y_{k+1} - y_k\bigr)$$
>
> Use step size $\alpha = 1/L$ and momentum $\gamma = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}$.

In [ ]:
def nesterov(A, b, x_star, alpha, mu, L, n_iter=500):
    """
    Nesterov Accelerated Gradient Descent for strongly-convex quadratics.

    Parameters
    ----------
    alpha : step size  (use 1/L)
    mu    : strong-convexity constant
    L     : smoothness constant
    """
    kappa = L / mu
    gamma = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)   # momentum

    y = np.zeros_like(b)        # current iterate
    y_prev = y.copy()           # previous iterate
    f_star = f(x_star, A, b)

    # We track the 'y' sequence which converges to x*
    losses = [f(y, A, b) - f_star]

    for _ in range(n_iter):
        # ── TODO ─────────────────────────────────────────────────────────────
        # x_k (the point where we evaluate the gradient) comes from adding
        # momentum to y:
        #   x = y + gamma * (y - y_prev)
        # Then the gradient step gives the new y:
        #   y_new = x - alpha * grad_f(x, A, b)
        #
        # x     = ???
        # y_new = ???
        # y_prev = ???
        # y      = ???
        # ─────────────────────────────────────────────────────────────────────
        losses.append(f(y, A, b) - f_star)

    return losses

---
## 3 — Convergence rates: theory vs. practice

The theoretical per-step contraction factors are:

| Method | Rate $\rho$ | $f(x_k)-f^\star \le$ |
|--------|-------------|----------------------|
| GD | $\dfrac{\kappa-1}{\kappa+1}$ | $\rho^k \cdot (f(x_0)-f^\star)$ |
| Heavy Ball | $\left(\dfrac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2$ | same |
| Nesterov | $\left(\dfrac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2$ | same |

Both accelerated methods achieve the same **asymptotic rate** on quadratics, but Nesterov is more robust off-quadratics.

> **✏️ Exercise 3.1** — Complete the helper below that computes the theoretical convergence curve.

In [ ]:
def theoretical_rate(rho, loss_0, n_iter):
    """
    Return the theoretical upper bound  loss_0 * rho^k  for k = 0 .. n_iter.

    Parameters
    ----------
    rho    : per-step contraction factor  (0 < rho < 1)
    loss_0 : initial suboptimality  f(x_0) - f*
    """
    # ── TODO ──────────────────────────────────────────────────────────────────
    # return ???   (a numpy array of length n_iter + 1)
    # ─────────────────────────────────────────────────────────────────────────
    pass


def gd_rate(kappa):
    """Contraction factor of GD at optimal step."""
    # ── TODO ──────────────────────────────────────────────────────────────────
    # return ???
    # ─────────────────────────────────────────────────────────────────────────
    pass


def accel_rate(kappa):
    """Contraction factor of HB / Nesterov at optimal parameters."""
    # ── TODO ──────────────────────────────────────────────────────────────────
    # return ???
    # ─────────────────────────────────────────────────────────────────────────
    pass

---
## 4 — Experiment A: fixed condition number, vary step size

> **✏️ Exercise 4** — Run GD with four different step sizes ($\alpha \in \{0.5/L,\; 1/L,\; 1.5/L,\; 1.9/L\}$) and plot the suboptimality on a log scale. 
>
> What happens when $\alpha > 2/L$?  Why?

In [ ]:
# ── Problem ──────────────────────────────────────────────────────────────────
d, kappa_val, n_iter = 50, 20, 300
A, b, x_star, mu, L = make_problem(d=d, kappa=kappa_val)

alphas = [0.5 / L, 1.0 / L, 1.5 / L, 1.9 / L]
labels = [r'$\alpha = 0.5/L$', r'$\alpha = 1/L$ (optimal)',
          r'$\alpha = 1.5/L$', r'$\alpha = 1.9/L$']

fig, ax = plt.subplots(figsize=(8, 4))

for alpha, label in zip(alphas, labels):
    # ── TODO ──────────────────────────────────────────────────────────────────
    # losses = gradient_descent(???)
    # ax.semilogy(losses, label=label)
    # ─────────────────────────────────────────────────────────────────────────
    pass

ax.set_xlabel('Iteration $k$')
ax.set_ylabel(r'$f(x_k) - f^\star$')
ax.set_title(f'GD — varying step size   ($\\kappa={kappa_val}$)')
ax.legend()
plt.tight_layout()

---
## 5 — Experiment B: compare all three methods

> **✏️ Exercise 5** — For $\kappa \in \{5, 50, 500\}$, run all three optimisers with their **optimal parameters** and plot the suboptimality curves together with the theoretical rates.

In [ ]:
kappas   = [5, 50, 500]
n_iter   = 600
d        = 100

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, kappa_val in zip(axes, kappas):
    A, b, x_star, mu, L = make_problem(d=d, kappa=kappa_val)

    # ── optimal parameters ───────────────────────────────────────────────────
    alpha_gd = 1.0 / L
    # Heavy Ball optimal params  (see Section 2)
    alpha_hb = (2 / (np.sqrt(L) + np.sqrt(mu))) ** 2
    beta_hb  = ((np.sqrt(kappa_val) - 1) / (np.sqrt(kappa_val) + 1)) ** 2

    # ── TODO: run the three methods ──────────────────────────────────────────
    # losses_gd  = gradient_descent(???)
    # losses_hb  = heavy_ball(???)
    # losses_nag = nesterov(???)
    # ─────────────────────────────────────────────────────────────────────────

    # ── TODO: theoretical bounds ─────────────────────────────────────────────
    # loss_0 = losses_gd[0]
    # theory_gd  = theoretical_rate(gd_rate(kappa_val),    loss_0, n_iter)
    # theory_acc = theoretical_rate(accel_rate(kappa_val), loss_0, n_iter)
    # ─────────────────────────────────────────────────────────────────────────

    # ── TODO: plot ───────────────────────────────────────────────────────────
    # ax.semilogy(losses_gd,  color='steelblue',  label='GD')
    # ax.semilogy(losses_hb,  color='tomato',     label='Heavy Ball')
    # ax.semilogy(losses_nag, color='seagreen',   label='Nesterov')
    # ax.semilogy(theory_gd,  color='steelblue',  ls='--', lw=1, label='GD theory')
    # ax.semilogy(theory_acc, color='seagreen',   ls='--', lw=1, label='Accel theory')
    # ─────────────────────────────────────────────────────────────────────────

    ax.set_xlabel('Iteration $k$')
    ax.set_ylabel(r'$f(x_k)-f^\star$')
    ax.set_title(f'$\\kappa = {kappa_val}$')
    ax.legend(fontsize=8)

plt.suptitle('GD vs Heavy Ball vs Nesterov — optimal parameters', y=1.02)
plt.tight_layout()

---
## 6 — Experiment C: sensitivity to the momentum parameter $\beta$

> **✏️ Exercise 6** — Fix $\kappa = 50$ and vary the momentum $\beta$ for the Heavy Ball method: 
> $\beta \in \{0,\; \beta^\star/2,\; \beta^\star,\; 0.99\}$.
>
> *Questions to think about:*
> - What does $\beta = 0$ reduce to?
> - What happens for very large $\beta$ (close to 1)?

In [ ]:
kappa_val = 50
A, b, x_star, mu, L = make_problem(d=100, kappa=kappa_val)

alpha_hb = (2 / (np.sqrt(L) + np.sqrt(mu))) ** 2
beta_opt = ((np.sqrt(kappa_val) - 1) / (np.sqrt(kappa_val) + 1)) ** 2

betas  = [0.0, beta_opt / 2, beta_opt, 0.99]
labels = [r'$\beta=0$ (GD)', r'$\beta=\beta^\star/2$',
          r'$\beta=\beta^\star$ (optimal)', r'$\beta=0.99$']

fig, ax = plt.subplots(figsize=(8, 4))

for beta, label in zip(betas, labels):
    # ── TODO ──────────────────────────────────────────────────────────────────
    # losses = heavy_ball(???)
    # ax.semilogy(losses, label=label)
    # ─────────────────────────────────────────────────────────────────────────
    pass

ax.set_xlabel('Iteration $k$')
ax.set_ylabel(r'$f(x_k) - f^\star$')
ax.set_title(f'Heavy Ball — varying $\\beta$   ($\\kappa={kappa_val}$)')
ax.legend()
plt.tight_layout()

---
## 7 — Experiment D: iterations to $\varepsilon$-accuracy vs. condition number

> **✏️ Exercise 7** — For each $\kappa \in \{2, 5, 10, 20, 50, 100, 200, 500\}$, record the number of iterations needed to reach $f(x_k) - f^\star \le 10^{-6}$ for all three methods. Plot on a log–log scale and fit the empirical scaling.
>
> **Expected slopes:** GD $\propto \kappa$, accelerated $\propto \sqrt{\kappa}$.

In [ ]:
kappas   = [2, 5, 10, 20, 50, 100, 200, 500]
eps      = 1e-6
n_iter   = 5000
d        = 80

iters_gd  = []
iters_hb  = []
iters_nag = []

def first_iter_below(losses, eps):
    """Return the first index k where losses[k] < eps, or len(losses)-1."""
    for k, v in enumerate(losses):
        if v < eps:
            return k
    return len(losses) - 1


for kappa_val in kappas:
    A, b, x_star, mu, L = make_problem(d=d, kappa=kappa_val)
    alpha_hb = (2 / (np.sqrt(L) + np.sqrt(mu))) ** 2
    beta_hb  = ((np.sqrt(kappa_val) - 1) / (np.sqrt(kappa_val) + 1)) ** 2

    # ── TODO ──────────────────────────────────────────────────────────────────
    # Run each method, compute first_iter_below, append to the three lists
    # ─────────────────────────────────────────────────────────────────────────
    pass


# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

# ── TODO ──────────────────────────────────────────────────────────────────────
# ax.loglog(kappas, iters_gd,  'o-', color='steelblue', label='GD')
# ax.loglog(kappas, iters_hb,  's-', color='tomato',    label='Heavy Ball')
# ax.loglog(kappas, iters_nag, '^-', color='seagreen',  label='Nesterov')
# Add reference lines proportional to kappa and sqrt(kappa)
# ─────────────────────────────────────────────────────────────────────────────

ax.set_xlabel(r'Condition number $\kappa$')
ax.set_ylabel(r'Iterations to $\varepsilon = 10^{-6}$')
ax.set_title('Iteration complexity vs condition number')
ax.legend()
plt.tight_layout()

---
## 8 — 2-D visualisation (bonus)

For a 2-D strongly convex quadratic it is illuminating to watch the **trajectory** of each method.

> **✏️ Exercise 8 (bonus)** — Adapt the code below to also record and plot the trajectory of the Heavy Ball method. 
> Observe the oscillatory behaviour when the condition number is large.

In [ ]:
def make_2d_problem(kappa):
    """A 2-D quadratic aligned with the axes for easy visualisation."""
    mu_val = 1.0
    L_val  = float(kappa)
    A = np.diag([mu_val, L_val])
    b = np.array([2.0, 1.0])
    x_star = np.linalg.solve(A, b)
    return A, b, x_star, mu_val, L_val


def run_2d(method, A, b, x_star, **kwargs):
    """Run a method and collect the iterate path."""
    x = np.array([-1.5, -1.5])
    path = [x.copy()]
    f_star = f(x_star, A, b)

    if method == 'gd':
        alpha = kwargs['alpha']
        for _ in range(kwargs.get('n_iter', 60)):
            x = x - alpha * grad_f(x, A, b)
            path.append(x.copy())

    elif method == 'nag':
        alpha, mu, L = kwargs['alpha'], kwargs['mu'], kwargs['L']
        kappa = L / mu
        gamma = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)
        y, y_prev = x.copy(), x.copy()
        for _ in range(kwargs.get('n_iter', 60)):
            xk = y + gamma * (y - y_prev)
            y_new = xk - alpha * grad_f(xk, A, b)
            y_prev, y = y, y_new
            path.append(y.copy())

    # ── TODO (bonus): add 'hb' branch here ───────────────────────────────────

    return np.array(path)


kappa_vis = 20
A2, b2, xs2, mu2, L2 = make_2d_problem(kappa_vis)

path_gd  = run_2d('gd',  A2, b2, xs2, alpha=1/L2, n_iter=80)
path_nag = run_2d('nag', A2, b2, xs2, alpha=1/L2, mu=mu2, L=L2, n_iter=80)

# ── contour plot ─────────────────────────────────────────────────────────────
xx = np.linspace(-2, 4, 300)
yy = np.linspace(-3, 3, 300)
XX, YY = np.meshgrid(xx, yy)
ZZ = np.array([[f(np.array([xi, yi]), A2, b2) for xi in xx] for yi in yy])

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(XX, YY, ZZ, levels=40, cmap='Blues', alpha=0.7)
ax.contour( XX, YY, ZZ, levels=40, colors='white', linewidths=0.4, alpha=0.5)
ax.plot(*xs2, 'k*', ms=12, zorder=5, label=r'$x^\star$')

ax.plot(path_gd[:,0],  path_gd[:,1],  'o-', color='tomato',   ms=3, lw=1.2, label='GD')
ax.plot(path_nag[:,0], path_nag[:,1], 's-', color='seagreen',  ms=3, lw=1.2, label='Nesterov')

ax.set_title(f'Trajectories in 2-D   ($\\kappa={kappa_vis}$)')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend()
plt.tight_layout()

---
## 9 — Discussion questions

Answer the following in the cells below.

1. **Step size sensitivity**: What is the maximum step size $\alpha$ for which GD converges? What happens at $\alpha = 2/L$?

2. **Acceleration gap**: By how many iterations does Nesterov beat GD for $\kappa = 100$ to reach $\varepsilon = 10^{-6}$? Does the gap match the theory ($\kappa$ vs $\sqrt{\kappa}$)?

3. **Heavy Ball vs Nesterov**: The two methods have the same asymptotic rate on quadratics. Can you think of a setting where one is preferred over the other?

4. **Oscillations**: When $\kappa$ is large, what do you observe in the 2-D trajectories? Why does Nesterov oscillate less than Heavy Ball?

*Your answers here.*